# Geometric Churn Model — Constant-Rate Retention Analysis

The **geometric (constant-rate) churn model** is the simplest probabilistic framework for customer retention. It assumes every customer faces the same, time-invariant churn probability $\theta$ at each period. The fraction of the original cohort still active at time $t$ follows:

$$S(t) = (1 - \theta)^t$$

The churn rate $\theta$ is estimated analytically via Maximum Likelihood Estimation (MLE):

$$\hat{\theta} = \frac{\text{total churners during calibration}}{\text{total customer-periods at risk}}$$

The model is intentionally simple and serves as a useful **baseline**. Because real cohorts exhibit heterogeneous churn risk — high-risk customers leave early, leaving a progressively more loyal survivor pool — the constant-rate model tends to over-predict churn at longer horizons. This notebook quantifies that effect and motivates more flexible approaches such as the Beta-Geometric model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Annual retention counts for a Standard customer segment (Years 0–12)
survivors = np.array([1000, 631, 468, 382, 326, 289, 262, 241, 223, 207, 194, 183, 173])
years     = np.arange(len(survivors))

df = pd.DataFrame({"Year": years, "Surviving Customers": survivors})
df

## Parameter Estimation

The model is calibrated on **Years 0–4** (the first four transitions). The MLE for a constant geometric churn rate has a closed-form solution: pool all churners and all customer-periods at risk over the calibration window.

The log-likelihood evaluates the probability of observing the exact sequence of dropout counts and final survivors:

$$\ell(\theta) = \sum_{t=1}^{T} d_t \ln p_t + N_T \ln\!\left(1 - \sum_{t=1}^{T} p_t\right)$$

where $d_t$ is the number of churners at period $t$, $p_t = \theta(1-\theta)^{t-1}$ is the probability of churning exactly at $t$, and $N_T$ is the number of surviving customers at the end of the calibration window.

In [ ]:
CALIB_END = 4  # calibrate on Years 0..4

at_risk = survivors[:CALIB_END]          # N_0, N_1, ..., N_{T-1}
churned = at_risk - survivors[1:CALIB_END + 1]  # d_1, d_2, ..., d_T

theta_hat     = churned.sum() / at_risk.sum()
retention_hat = 1.0 - theta_hat


def geometric_loglik(theta, churned, n_survivors_final, T):
    """Log-likelihood for the constant geometric churn model."""
    eps   = 1e-15
    probs = np.array([theta * (1 - theta) ** (t - 1) for t in range(1, T + 1)])
    probs = np.clip(probs, eps, 1 - eps)
    prob_survive = float(np.clip(1.0 - probs.sum(), eps, 1 - eps))
    return float((churned * np.log(probs)).sum()) + n_survivors_final * np.log(prob_survive)


ll = geometric_loglik(theta_hat, churned, survivors[CALIB_END], T=CALIB_END)

print(f"Estimated churn rate  θ̂ : {theta_hat:.6f}  ({theta_hat:.1%} per year)")
print(f"Estimated retention 1-θ̂ : {retention_hat:.6f}  ({retention_hat:.1%} per year)")
print(f"Log-likelihood (Years 0–{CALIB_END}): {ll:.4f}")

## Survival Forecast

Using $\hat{\theta}$, the survival curve $S(t) = (1 - \hat{\theta})^t$ is projected over the full 12-year horizon.

In [ ]:
df["S(t)"]               = retention_hat ** df["Year"]
df["Forecasted Survivors"] = (survivors[0] * df["S(t)"]).round(1)
df["Retention Rate"]     = (df["S(t)"] * 100).round(1).astype(str) + "%"

display_cols = ["Year", "Surviving Customers", "Forecasted Survivors", "Retention Rate"]
df[display_cols]

## Forecast Accuracy

Mean Absolute Percentage Error (MAPE) is computed separately for the **in-sample** window (Years 1–4, within the calibration period) and the **out-of-sample** window (Years 5–12, the hold-out period).

In [ ]:
actual   = df["Surviving Customers"].to_numpy(float)
forecast = df["Forecasted Survivors"].to_numpy(float)
ape      = np.abs((actual - forecast) / actual)

in_mape  = ape[1 : CALIB_END + 1].mean() * 100
out_mape = ape[CALIB_END + 1 :].mean()   * 100

print(f"In-sample MAPE  (Years 1–{CALIB_END}):   {in_mape:.1f}%")
print(f"Out-of-sample MAPE (Years {CALIB_END + 1}–12): {out_mape:.1f}%")

df["APE"] = (ape * 100).round(1).astype(str) + "%"
df[["Year", "Surviving Customers", "Forecasted Survivors", "APE"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Actual vs Forecast ---
ax = axes[0]
ax.plot(years, survivors,             "ko-", label="Actual",         linewidth=2, zorder=3)
ax.plot(years, forecast,              "b--", label="Model Forecast",  linewidth=1.8)
ax.axvspan(0, CALIB_END, alpha=0.07, color="steelblue", label="Calibration window")
ax.set_title("Actual vs. Forecast Survivors", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Active Customers")
ax.legend()

# --- Right: Survival curve as % of cohort ---
ax = axes[1]
ax.plot(years, actual / survivors[0] * 100,   "ko-", label="Actual",   linewidth=2, zorder=3)
ax.plot(years, forecast / survivors[0] * 100, "b--", label="Geometric", linewidth=1.8)
ax.axvspan(0, CALIB_END, alpha=0.07, color="steelblue")
ax.set_title("Survival Curve  $S(t) = (1 - \\hat{\\theta})^t$", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("% of Initial Cohort Surviving")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.legend()

fig.suptitle("Geometric Churn Model — Standard Segment (Calibration: Years 0–4)", fontsize=13)
plt.tight_layout()
plt.show()

## Key Findings

| Metric | Value |
|--------|-------|
| Estimated annual churn rate $\hat{\theta}$ | ~27.2% |
| Estimated annual retention $1 - \hat{\theta}$ | ~72.8% |
| In-sample MAPE (Years 1–4) | ~10.9% |
| Out-of-sample MAPE (Years 5–12) | ~64.0% |

The model fits the calibration window reasonably well but deteriorates sharply beyond it. The root cause is the **constant-rate assumption**: in reality, customers with high churn propensity leave early, so the surviving pool at later periods is increasingly composed of loyal customers — meaning the observed churn rate naturally declines over time. A model that assumes a fixed $\theta$ cannot capture this and systematically over-predicts dropout at longer horizons.

This divergence motivates the **Beta-Geometric model**, which introduces heterogeneity in $\theta$ across customers. By allowing $\theta \sim \text{Beta}(a, b)$, the BG model implicitly captures the survivor effect and produces substantially more accurate long-run forecasts on the same dataset.